# 第6章：LangChain 高级

> 🎯 掌握RouterChain、Callback、调试技巧

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
llm = ChatOpenAI(
    base_url=os.getenv('LM_STUDIO_BASE_URL', 'http://localhost:1234/v1'),
    api_key='lm-studio',
    model=os.getenv('LM_STUDIO_MODEL', 'qwen2.5-7b-instruct')
)

## 1. RouterChain 智能路由

In [ ]:
from langchain_core.runnables import RunnableBranch

tech_prompt = ChatPromptTemplate.from_template('作为技术专家回答：{question}')
life_prompt = ChatPromptTemplate.from_template('作为生活顾问回答：{question}')
default_prompt = ChatPromptTemplate.from_template('请回答：{question}')

def classify(x):
    q = x['question'].lower()
    if any(kw in q for kw in ['代码', 'python', '编程', 'api']):
        return 'tech'
    elif any(kw in q for kw in ['健康', '运动', '饮食']):
        return 'life'
    return 'other'

router = RunnableBranch(
    (lambda x: classify(x) == 'tech', tech_prompt | llm | StrOutputParser()),
    (lambda x: classify(x) == 'life', life_prompt | llm | StrOutputParser()),
    default_prompt | llm | StrOutputParser()
)

for q in ['Python怎么读文件？', '怎么保持健康？', '今天星期几？']:
    print(f'问: {q}')
    print(f'类型: {classify({"question": q})}')
    print(f'答: {router.invoke({"question": q})[:50]}...\n')

## 2. Callback 回调系统

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler
from datetime import datetime

class TimingCallback(BaseCallbackHandler):
    def on_llm_start(self, *args, **kwargs):
        self.start = datetime.now()
        print('🚀 LLM开始执行...')
    
    def on_llm_end(self, *args, **kwargs):
        duration = (datetime.now() - self.start).total_seconds()
        print(f'✅ 完成，耗时: {duration:.2f}秒')

chain = ChatPromptTemplate.from_template('讲个{topic}笑话') | llm | StrOutputParser()
result = chain.invoke({'topic': '程序员'}, config={'callbacks': [TimingCallback()]})
print(result)

## 3. 调试技巧

In [ ]:
from langchain_core.runnables import RunnableLambda

def debug_print(x):
    print(f'[DEBUG] {type(x).__name__}: {str(x)[:100]}...')
    return x

debug_chain = (
    ChatPromptTemplate.from_template('解释{concept}')
    | RunnableLambda(debug_print)
    | llm
    | RunnableLambda(debug_print)
    | StrOutputParser()
)

result = debug_chain.invoke({'concept': '闭包'})

## 📝 小结
1. **RouterChain**：`RunnableBranch` 智能路由
2. **Callback**：监控执行过程
3. **调试**：中间结果打印